# PGM Forcing Generator (Native Tool)

This notebook implements a **native forcing generator** for non-climate forcings without using MIKE ToolboxShell.

What it does:
- Define conversion jobs as tuples with:
  - setup name
  - input grid-code DFS2 path
  - per-grid-code input timeseries (DFS0 or CSV)
  - output DFS2 path
  - output item metadata (name, EUMType, EUMUnit)
- Read grid codes from DFS2
- Read and standardize all input timeseries
- Apply timeseries values to all cells sharing each grid code
- Write time-varying DFS2 output directly with `mikeio`

Rules:
- Missing timeseries for any grid code in the template grid => error
- Missing timesteps in each series are backfilled after alignment
- Near-daily timestamps with minor hour noise are normalized to daily to avoid excessive timesteps

Notes:
- Climate forcings remain out of scope here (provided through PDP repository).
- Paths may be absolute or relative to the module root.

## Step 1: Environment and Module Import

This step configures notebook-relative paths and imports the native forcing-generator module from src.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import mikeio

REPO_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
SRC_DIR = REPO_ROOT.joinpath("src")

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from plant_growth_module import forcing_generator_native  # noqa: E402

MODULE_ROOT = REPO_ROOT
print("Repo root:   ", REPO_ROOT)

## Step 2: Configure Inputs and Outputs

Set user-editable paths and metadata for one forcing generation run:
- Grid-code DFS2
- Per-grid-code timeseries inputs (DFS0/CSV)
- Output DFS2 and item metadata

Tip: If you are unsure which EUM type or EUM unit values are valid, use the utility section further down in this notebook (Utilities: EUM Type and Unit Lookup).

In [ ]:
OUTPUT_BASE = MODULE_ROOT.joinpath(
    "output_data", "pgm_forcing_generator", "forcing_generator"
)

# Single setup configuration for native grid-code forcing generation.
GRID_CODE_DFS2 = MODULE_ROOT.joinpath(
    "sample_data",
    "pgm_forcing_generator",
    "example_dfs0_dfs2",
    "GridCode5.dfs2",
)

TIMESERIES_INPUTS = [
    {
        "grid_code": 1,
        "path": MODULE_ROOT.joinpath(
            "sample_data",
            "pgm_forcing_generator",
            "example_dfs0_dfs2",
            "DMI_Jutland_2011-2023_item1.csv",
        ),
        "source": "dfs0",
    },
    {
        "grid_code": 5,
        "path": MODULE_ROOT.joinpath(
            "sample_data",
            "pgm_forcing_generator",
            "example_dfs0_dfs2",
            "DMI_Jutland_2011-2023.dfs0",
        ),
        "source": "dfs0",
        "item": 5,
    },
]

OUTPUT_GRID = OUTPUT_BASE.joinpath("Jytland-test-ouput.dfs2")


OUTPUT_ITEM_NAME = "Seed Application Rate"
OUTPUT_EUM_TYPE = "Concentration"
OUTPUT_EUM_UNIT = "kg_per_meter_pow_3"

# Quick checks for configured output and setup fields.
print("Forcing generator output base:   ", OUTPUT_BASE)
print("Grid code DFS2:                  ", GRID_CODE_DFS2)
print("Number of timeseries inputs:     ", len(TIMESERIES_INPUTS))
print("Output grid:                     ", OUTPUT_GRID)
print("Output item name:                ", OUTPUT_ITEM_NAME)
print("Output EUM type:                 ", OUTPUT_EUM_TYPE)
print("Output EUM unit:                 ", OUTPUT_EUM_UNIT)

## Step 3: Validate Configuration

Resolve absolute paths, check required files/folders, and print the concrete run inputs.

In [ ]:
grid_path = forcing_generator_native.to_abs_path(MODULE_ROOT, GRID_CODE_DFS2)
output_grid = forcing_generator_native.to_abs_path(MODULE_ROOT, OUTPUT_GRID)
grid_exists = grid_path.exists()
output_parent_exists = output_grid.parent.exists()

input_rows = forcing_generator_native.build_input_rows(MODULE_ROOT, TIMESERIES_INPUTS)

print("Grid-code DFS2:          ", grid_path)
print("Grid-code DFS2 exists:   ", grid_exists)
print("Output parent exists:    ", output_parent_exists)
print("Output item name:        ", OUTPUT_ITEM_NAME)
print("Output EUM type:         ", OUTPUT_EUM_TYPE)
print("Output EUM unit:         ", OUTPUT_EUM_UNIT)
print()
print("Timeseries inputs:")
for row in input_rows:
    print(
        f"  grid_code={row['grid_code']}, source={row['source']}, "
        f"item={row['item']}, exists={row['path_exists']}"
    )
    print(f"    path: {row['path']}")

## Step 4: Run Native Forcing Generation

Execute the native generator and write the output DFS2 grid series.

In [ ]:
invalid_inputs = [row for row in input_rows if not row["path_exists"]]

if not grid_exists:
    raise FileNotFoundError(f"Grid-code DFS2 file not found: {grid_path}")

if invalid_inputs:
    details = "\n".join(
        f"grid_code={row['grid_code']}, source={row['source']}, path={row['path']}"
        for row in invalid_inputs
    )
    raise FileNotFoundError("Missing timeseries input files:\n" + details)

run_result = forcing_generator_native.run_native_setup(
    module_root=MODULE_ROOT,
    grid_code_dfs2=GRID_CODE_DFS2,
    timeseries_inputs=TIMESERIES_INPUTS,
    output_grid=OUTPUT_GRID,
    output_item_name=OUTPUT_ITEM_NAME,
    output_eum_type=OUTPUT_EUM_TYPE,
    output_eum_unit=OUTPUT_EUM_UNIT,
)

print("Native forcing generation completed.")
print("  Output:      ", run_result["output_grid"])
print("  Timesteps:   ", run_result["n_timesteps"])
print("  Time range:  ", run_result["start_time"], "->", run_result["end_time"])

## Step 5: Inspect Run Outputs

Read the generated DFS2 and print key output diagnostics.

### Step 5.A: Output Summary

Run the next cell to inspect written DFS2 metadata and integrity details.

In [ ]:
output_path = Path(run_result["output_grid"])
output_exists = output_path.exists()

if output_exists:
    generated = mikeio.read(output_path)
    generated_da = generated[0]
    written_item_name = generated_da.item.name
    written_timesteps = len(generated_da.time)
else:
    written_item_name = "<missing>"
    written_timesteps = 0

print("Run result summary:")
print("  Grid code DFS2:        ", run_result["grid_code_dfs2"])
print("  Output grid:           ", run_result["output_grid"])
print("  Number of grid codes:  ", run_result["n_grid_codes"])
print("  Timesteps:             ", run_result["n_timesteps"])
print("  Start time:            ", run_result["start_time"])
print("  End time:              ", run_result["end_time"])
print("  Output exists:         ", output_exists)
print("  Written item name:     ", written_item_name)
print("  Written timesteps:     ", written_timesteps)

## Utilities: EUM Type and Unit Lookup

Use this utility to discover valid `OUTPUT_EUM_TYPE` and `OUTPUT_EUM_UNIT` values before running Step 4.

What this helps with:
- Find candidate EUM types by text search
- Find candidate EUM units by text search
- Inspect compatibility between a specific type and unit

How to use:
- Set `EUM_TYPE_QUERY` and/or `EUM_UNIT_QUERY` in the next cell
- Run the next cell to print matching names
- If an exact type/unit is provided, the utility also prints compatible counterparts

### Utility Notes

Suggested workflow:
- Start with a broad query, for example `Concentration` or `kg`
- Copy one exact match into your Step 2 configuration
- Re-run this utility to confirm compatibility when in doubt

Output interpretation:
- `Matched EUM types`: names containing `EUM_TYPE_QUERY`
- `Matched EUM units`: names containing `EUM_UNIT_QUERY`
- `Compatible units`: units valid for an exact type match
- `Compatible types`: types valid for an exact unit match

In [ ]:
# EUM helper: search possible EUM types/units and inspect compatibility.
# Set one or both queries below, then run this cell.
EUM_TYPE_QUERY = ""  # example: "Concentration"
EUM_UNIT_QUERY = ""  # example: "kg_per_meter_pow_3"

matches = forcing_generator_native.eum_matches(
    type_query=EUM_TYPE_QUERY,
    unit_query=EUM_UNIT_QUERY,
)

matched_types = matches["matched_types"]
matched_units = matches["matched_units"]
compatible_units = matches["compatible_units"]
compatible_types = matches["compatible_types"]

print(f"Matched EUM types ({len(matched_types)}):")
print(matched_types[:100])
if len(matched_types) > 100:
    print("... truncated ...")

print()
print(f"Matched EUM units ({len(matched_units)}):")
print(matched_units[:100])
if len(matched_units) > 100:
    print("... truncated ...")

if compatible_units:
    print()
    print(f"Compatible units ({len(compatible_units)}):")
    print(compatible_units)

if compatible_types:
    print()
    print(f"Compatible types ({len(compatible_types)}):")
    print(compatible_types)